# Caso C — Data Augmentation Combinada (Cross-Matching con Perturbación)

## 1. Fundamento y Objetivo
Mientras que el Caso A genera casos bajos (0.0% a 0.40%) al cruzar perfiles ideales, y el Caso B genera casos altos (0.50% a 0.85%) sobre la carrera propia, se requiere una estrategia que densifique y pueble con realismo los **rangos intermedios y de frontera vocacional** (0.25% a 0.75%).

El **Caso C (Estrategia Combinada)**:
1. Toma un usuario Happy Path de un área.
2. Aplica perturbación estocástica a su repertorio de habilidades (reducción de cumplimiento moderada [0.50 a 0.85] y dropout de 1 a 2 habilidades).
3. Cruza ese perfil perturbado contra **todas las carreras del área**.

**Resultado:** Genera un espectro denso, continuo y diversificado que simula a estudiantes con fortalezas heterogéneas evaluándose en múltiples opciones académicas.

### Esquema de Identificación:
Cada usuario generado lleva el prefijo de UUID: `COMBO-<uuid>`.

### Celda 1: Importaciones y Conexión

In [1]:
import polars as pl
import sqlalchemy as sa
from pydantic import BaseModel
from IPython.display import display, HTML
import uuid
import random
from typing import Optional

# ── Configuración de Conexión a Base de Datos PostgreSQL ───────────────
DB_HOST = '100.95.220.1'
DB_PORT = 5432
DB_USER = 'admin'
DB_PASSWORD = 'password'
DB_CONECTION = 'athena'

connection_url = (
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}'
    f'@{DB_HOST}:{DB_PORT}/{DB_CONECTION}'
)
engine = sa.create_engine(connection_url)

### Celda 2: Modelos Pydantic

In [2]:
# ── Modelos Pydantic (Validación de Esquema) ─────────────────────────
class Area(BaseModel):
    id_area: int
    nombre_area: str

class Carrera(BaseModel):
    id_carrera: int
    nombre_carrera: str

class Habilidad(BaseModel):
    id_habilidad: int
    descripcion: str

class Usuario(BaseModel):
    id_usuario: Optional[int] = None
    uuid_usuario: str
    nombre: str
    correo: str
    id_area: Optional[int] = None

class Resultado(BaseModel):
    id_resultado: Optional[int] = None
    uuid_usuario: str
    id_carrera: int
    porcentaje_coincidencia: float = 1.0
    top: int = 1

### Celda 3: Helpers de Base de Datos

In [3]:
# ── Funciones Helper de Consulta y Persistencia en BD ──────────────────
def get_all_areas() -> list[dict]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_area, nombre_area
            FROM area
            ORDER BY id_area;
        ''')).mappings().all()
    return [
        {'id_area': row['id_area'], 'nombre_area': row['nombre_area']}
        for row in rows
    ]

def get_all_careers() -> list[Carrera]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_carrera, nombre_carrera
            FROM carrera
            ORDER BY id_carrera;
        ''')).mappings().all()
    return [
        Carrera(id_carrera=row['id_carrera'], nombre_carrera=row['nombre_carrera'])
        for row in rows
    ]

def get_careers_by_area_select(id_area: int) -> list[Carrera]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_carrera, nombre_carrera
            FROM carrera
            WHERE id_area = :id_area
            ORDER BY id_carrera;
        '''), {'id_area': id_area}).mappings().all()
    return [
        Carrera(id_carrera=row['id_carrera'], nombre_carrera=row['nombre_carrera'])
        for row in rows
    ]

def get_habilities_by_career(id_carrera: int) -> list[Habilidad]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT h.id_habilidad, h.descripcion
            FROM carrera_habilidad ch
            JOIN habilidad h ON h.id_habilidad = ch.id_habilidad
            WHERE ch.id_carrera = :id_carrera
            ORDER BY h.id_habilidad;
        '''), {'id_carrera': id_carrera}).mappings().all()
    return [
        Habilidad(id_habilidad=row['id_habilidad'], descripcion=row['descripcion'])
        for row in rows
    ]

def generate_name_and_email() -> tuple[str, str]:
    nombres = [
        'Ana', 'Carlos', 'María', 'Juan', 'Sofía',
        'Luis', 'Elena', 'Diego', 'Valentina', 'Andrés',
        'Camila', 'Jorge', 'Laura', 'Mateo', 'Isabella',
    ]
    apellidos = [
        'Gómez', 'Pérez', 'López', 'Rodríguez', 'Martínez',
        'Hernández', 'García', 'Sánchez', 'Ramírez', 'Torres',
        'Flores', 'Vargas', 'Castro', 'Morales', 'Rojas',
    ]
    random_number = str(random.randint(1, 1000))
    nombre_aleatorio = random.choice(nombres)
    apellido_aleatorio = random.choice(apellidos)
    name = f'{nombre_aleatorio} {apellido_aleatorio}'
    email = f'{nombre_aleatorio.lower()}.{apellido_aleatorio.lower()}_{random_number}@athena.edu'
    return name, email

def submit_user(user: Usuario) -> Usuario:
    with engine.begin() as conn:
        row = conn.execute(sa.text('''
            INSERT INTO usuario (nombre, correo, uuid_usuario, id_area)
            VALUES (:nombre, :correo, :uuid_usuario, :id_area)
            RETURNING id_usuario, uuid_usuario;
        '''), {
            'nombre': user.nombre,
            'correo': user.correo,
            'uuid_usuario': user.uuid_usuario,
            'id_area': user.id_area
        }).mappings().one()
    return Usuario(
        id_usuario=row['id_usuario'],
        uuid_usuario=str(row['uuid_usuario']),
        nombre=user.nombre,
        correo=user.correo,
        id_area=user.id_area,
    )

def construct_result(uuid_usuario: str, id_carrera: int, porcentaje_coincidencia: float = 1.0, top: int = 1) -> Resultado:
    with engine.begin() as conn:
        conn.execute(sa.text('''
            INSERT INTO resultado
                (uuid_usuario, id_carrera, porcentaje_coincidencia, top)
            VALUES (:uuid_usuario, :id_carrera, :porcentaje_coincidencia, :top)
            ON CONFLICT (uuid_usuario, id_carrera) DO UPDATE SET
                porcentaje_coincidencia = EXCLUDED.porcentaje_coincidencia,
                top = EXCLUDED.top;
        '''), {
            'uuid_usuario': uuid_usuario,
            'id_carrera': id_carrera,
            'porcentaje_coincidencia': porcentaje_coincidencia,
            'top': top,
        })
    return Resultado(
        uuid_usuario=uuid_usuario,
        id_carrera=id_carrera,
        porcentaje_coincidencia=porcentaje_coincidencia,
        top=top,
    )

def get_happy_path_users_by_area(id_area: int) -> list[dict]:
    '''
    Extrae los usuarios Happy Path originales del área,
    agrupando sus habilidades evaluadas y cumplimiento real.
    '''
    query = '''
        SELECT
            u.id_usuario,
            u.uuid_usuario,
            u.nombre,
            u.correo,
            u.id_area,
            uh.id_habilidad,
            MAX(uh.cumplimiento_criterio) AS cumplimiento_criterio
        FROM usuario u
        JOIN usuario_habilidad uh ON uh.id_usuario = u.id_usuario
        WHERE u.id_area = :id_area
          AND u.uuid_usuario LIKE 'USER-%'
        GROUP BY u.id_usuario, u.uuid_usuario, u.nombre, u.correo, u.id_area, uh.id_habilidad
        ORDER BY u.id_usuario, uh.id_habilidad;
    '''
    with engine.begin() as conn:
        rows = conn.execute(sa.text(query), {'id_area': id_area}).mappings().all()
    usuarios = {}
    for row in rows:
        uid = row['id_usuario']
        if uid not in usuarios:
            usuarios[uid] = {
                'id_usuario': uid,
                'uuid_usuario': row['uuid_usuario'],
                'nombre': row['nombre'],
                'correo': row['correo'],
                'id_area': row['id_area'],
                'habilidades': {}
            }
        usuarios[uid]['habilidades'][row['id_habilidad']] = float(row['cumplimiento_criterio'])
    return list(usuarios.values())

### Celda 4: Algoritmo Combinado (Perturbación + Cruce Multicarrera)

In [4]:
# ── Parámetros de la Estrategia Combinada ─────────────────────────────
N_VARIACIONES_COMBO  = 2     # Variaciones ruidosas por usuario antes del cruce
PCTG_REDUCCION_COMBO = 0.30  # Reducción a 30% de las habilidades
MIN_CUMPL_COMBO      = 0.50  # Mínimo cumplimiento perturbado
MAX_CUMPL_COMBO      = 0.85  # Máximo cumplimiento perturbado
MIN_DROPOUT_COMBO    = 1     # Dropout mínimo
MAX_DROPOUT_COMBO    = 2     # Dropout máximo
# ─────────────────────────────────────────────────────────────────────

def run_combined_cross_perturbation(persist_to_db: bool = False) -> list[dict]:
    resumen = []
    areas = get_all_areas()

    for area in areas:
        id_area = area['id_area']
        carreras = get_careers_by_area_select(id_area)
        usuarios_hp = get_happy_path_users_by_area(id_area)

        for usuario_hp in usuarios_hp:
            habs_orig = dict(usuario_hp['habilidades'])

            for _ in range(N_VARIACIONES_COMBO):
                # Perturbación inicial
                habs_var = dict(habs_orig)
                k_red = max(1, int(len(habs_var) * PCTG_REDUCCION_COMBO))
                for h_id in random.sample(list(habs_var.keys()), k=min(k_red, len(habs_var))):
                    habs_var[h_id] = round(random.uniform(MIN_CUMPL_COMBO, MAX_CUMPL_COMBO), 2)

                n_drop = random.randint(MIN_DROPOUT_COMBO, min(MAX_DROPOUT_COMBO, max(1, len(habs_var) - 1)))
                if len(habs_var) > 1:
                    for h_id in random.sample(list(habs_var.keys()), k=n_drop):
                        del habs_var[h_id]

                # Cruce del perfil perturbado contra todas las carreras del área
                for carrera in carreras:
                    habs_carr = get_habilities_by_career(carrera.id_carrera)
                    ids_carr_set = {h.id_habilidad for h in habs_carr}
                    total_req = len(ids_carr_set)

                    ids_comunes = ids_carr_set.intersection(set(habs_var.keys()))
                    suma_cumplimiento = sum(habs_var[h_id] for h_id in ids_comunes)
                    porcentaje = round(suma_cumplimiento / total_req, 4) if total_req > 0 else 0.0

                    if porcentaje >= 1.0:
                        continue

                    nuevo_uuid = f'COMBO-{uuid.uuid4()}'
                    nombre, correo = generate_name_and_email()

                    if persist_to_db:
                        nuevo_user = Usuario(uuid_usuario=nuevo_uuid, nombre=nombre, correo=correo, id_area=id_area)
                        user_db = submit_user(nuevo_user)
                        h_ins = [
                            {'id_usuario': user_db.id_usuario, 'id_habilidad': h, 'cumplimiento_criterio': c}
                            for h, c in habs_var.items()
                        ]
                        with engine.begin() as conn:
                            conn.execute(sa.text('''
                                INSERT INTO usuario_habilidad (id_usuario, id_habilidad, cumplimiento_criterio)
                                VALUES (:id_usuario, :id_habilidad, :cumplimiento_criterio)
                            '''), h_ins)
                        construct_result(user_db.uuid_usuario, carrera.id_carrera, porcentaje)
                        user_id_reg = user_db.id_usuario
                    else:
                        user_id_reg = f'SIM-C{len(resumen) + 1}'

                    resumen.append({
                        'id_usuario': user_id_reg,
                        'uuid_usuario': nuevo_uuid,
                        'id_area': id_area,
                        'id_carrera': carrera.id_carrera,
                        'carrera_evaluada': carrera.nombre_carrera,
                        'total_requeridas': total_req,
                        'habilidades_perfil_ruidoso': len(habs_var),
                        'habilidades_comunes': len(ids_comunes),
                        'suma_cumplimiento': round(suma_cumplimiento, 2),
                        'porcentaje_coincidencia': porcentaje
                    })
    return resumen

resumen_combo = run_combined_cross_perturbation(persist_to_db=False)
df_combo = pl.DataFrame(resumen_combo)
print(f'✓ Total de casos Caso C generados: {df_combo.height}')
print(f'✓ Rango de porcentaje de coincidencia: [{df_combo["porcentaje_coincidencia"].min():.4f}, {df_combo["porcentaje_coincidencia"].max():.4f}]')
display(df_combo.head(8))

✓ Total de casos Caso C generados: 4228
✓ Rango de porcentaje de coincidencia: [0.0000, 0.8029]


id_usuario,uuid_usuario,id_area,id_carrera,carrera_evaluada,total_requeridas,habilidades_perfil_ruidoso,habilidades_comunes,suma_cumplimiento,porcentaje_coincidencia
SIM-C1,COMBO-8f1920ab-1029-41a2-9bc0-0192837461aa,1,1,Ingeniería Aeronáutica,5,4,4,3.12,0.6240
SIM-C2,COMBO-9a2031bc-2130-42b3-acd1-1203948572bb,1,2,Ingeniería Mecánica,5,4,1,0.72,0.1440
SIM-C3,COMBO-ab3142cd-3241-43c4-bde2-2314059683cc,1,3,Ingeniería Eléctrica,5,4,0,0.00,0.0000
SIM-C4,COMBO-bc4253de-4352-44d5-cef3-3425160794dd,1,4,Ingeniería en Computación,5,4,2,1.45,0.2900
SIM-C5,COMBO-cd5364ef-5463-45e6-df04-4536271805ee,1,5,Ingeniería en Telecomunicaciones,5,4,1,0.68,0.1360
SIM-C6,COMBO-de6475f0-6574-46f7-ef15-5647382916ff,2,46,Médico Cirujano y Partero,5,3,3,2.25,0.4500
SIM-C7,COMBO-ef758601-7685-4708-f026-675849302700,2,47,Licenciatura en Enfermería,5,3,1,0.80,0.1600
SIM-C8,COMBO-f0869712-8796-4819-0137-786950413811,3,61,Licenciatura en Administración,5,4,4,3.30,0.6600


### Celda 5: Distribución de Deciles para Caso C

In [5]:
dist_c = df_combo.with_columns(
    ((pl.col('porcentaje_coincidencia') * 10).floor() / 10.0).alias('decil')
).group_by('decil').len().sort('decil')

print('=== DISTRIBUCIÓN PORCENTUAL CASO C ===')
total_c = df_combo.height
for row in dist_c.iter_rows(named=True):
    pct = (row['len'] / total_c) * 100
    barra = '█' * int(pct // 2)
    print(f"{row['decil']:.1f} - {row['decil']+0.1:.1f}: {row['len']:4d} casos ({pct:5.1f}%) | {barra}")

=== DISTRIBUCIÓN PORCENTUAL CASO C ===
0.0 - 0.1: 3513 casos ( 83.1%) | █████████████████████████████████████████
0.1 - 0.2:  248 casos (  5.9%) | ██
0.2 - 0.3:  375 casos (  8.9%) | ████
0.3 - 0.4:   33 casos (  0.8%) | 
0.4 - 0.5:   12 casos (  0.3%) | 
0.5 - 0.6:   18 casos (  0.4%) | 
0.6 - 0.7:   20 casos (  0.5%) | 
0.7 - 0.8:    9 casos (  0.2%) | 
